# Recurrence Basis — 3D Visualization

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

torch.manual_seed(0)
np.random.seed(0)

# ---------------------------------------------------------------------------
# Shared learnable recurrence basis:
#   R_0 = 0, R_1 = 1
#   R_{n+1} = (a h^2 + b h + c) R_n + (d h + e) R_{n-1}
#   a, b, c, d, e -> ONE set of parameters shared across the entire network
# ---------------------------------------------------------------------------
class RecurrenceBasis(nn.Module):
    def __init__(self, degree, init=(0.0, 2.0, 0.0, -1.0, 0.0), normalize=True):
        super().__init__()
        self.degree = degree
        self.normalize = normalize
        a, b, c, d, e = init
        self.a = nn.Parameter(torch.tensor(float(a)))
        self.b = nn.Parameter(torch.tensor(float(b)))
        self.c = nn.Parameter(torch.tensor(float(c)))
        self.d = nn.Parameter(torch.tensor(float(d)))
        self.e = nn.Parameter(torch.tensor(float(e)))

        # lets the training loop freeze/unfreeze a..e during warm-up
        self.recurrence_params = [self.a, self.b, self.c, self.d, self.e]

    def forward(self, h):
        coef2 = self.a * h ** 2 + self.b * h + self.c
        coef1 = self.d * h + self.e

        R_prev2 = torch.zeros_like(h)   # R_0
        R_prev1 = torch.ones_like(h)    # R_1
        basis = [R_prev2, R_prev1]
        for _ in range(self.degree - 1):
            R_next = coef2 * R_prev1 + coef1 * R_prev2

            if self.normalize:
                # keep magnitude of successive R_n under control using a
                # single GLOBAL scalar (not per-feature!). Reducing over
                # dim=-1 (the input_dim/feature axis) would mix unrelated
                # features (e.g. x and y) together and destroy information
                # — that was the bug. A scalar keeps every feature's
                # relative scale intact while still preventing blow-up.
                scale = R_next.detach().abs().amax().clamp(min=1e-3)
                R_next = R_next / scale

            basis.append(R_next)
            R_prev2, R_prev1 = R_prev1, R_next
        return torch.stack(basis, dim=-1)   # (..., degree+1)


class RecKANLayer(nn.Module):
    def __init__(self, input_dim, output_dim, degree, basis: RecurrenceBasis):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.degree = degree
        self.basis = basis  # shared instance across all layers/network
        self.coeffs = nn.Parameter(
            torch.randn(input_dim, output_dim, degree + 1)
            / (input_dim * (degree + 1)) ** 0.5
        )

    def forward(self, x):
        x = torch.tanh(x)                 # normalize to (-1, 1)
        R = self.basis(x)                  # (batch, input_dim, degree+1)
        y = torch.einsum('bid,iod->bo', R, self.coeffs)
        return y


# 3-layer RecKAN mirroring the ChebyKAN architecture (2 -> 8 -> 16 -> 1)
class RecKAN(nn.Module):
    def __init__(self, degree=8, normalize=True):
        super().__init__()
        self.basis = RecurrenceBasis(degree, normalize=normalize)   # shared a,b,c,d,e
        self.l1 = RecKANLayer(2, 8, degree, self.basis)
        self.l2 = RecKANLayer(8, 16, degree, self.basis)
        self.l3 = RecKANLayer(16, 1, degree, self.basis)

    def forward(self, x):
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        return x

    def recurrence_parameters(self):
        return self.basis.recurrence_params

    def coeff_parameters(self):
        return [self.l1.coeffs, self.l2.coeffs, self.l3.coeffs]

    def set_recurrence_trainable(self, flag: bool):
        for p in self.recurrence_parameters():
            p.requires_grad_(flag)


# ---------------------------------------------------------------------------
# Baseline MLP (same as your reference script)
# ---------------------------------------------------------------------------
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.layers(x)


# ---------------------------------------------------------------------------
# Same fractal-like noisy 2D target function
# ---------------------------------------------------------------------------
def fractal_function(x, y):
    z = np.sin(10 * np.pi * x) * np.cos(10 * np.pi * y) + np.sin(np.pi * (x ** 2 + y ** 2))
    z += np.abs(x - y) + (np.sin(5 * x * y) / (0.1 + np.abs(x + y)))
    z *= np.exp(-0.1 * (x ** 2 + y ** 2))
    noise = np.random.normal(0, 0.1, z.shape)
    z += noise
    return z


x = np.linspace(0, 2, 100)
y = np.linspace(0, 2, 100)
X, Y = np.meshgrid(x, y)
Z = fractal_function(X, Y)
x_train_2d = torch.tensor(np.stack([X.ravel(), Y.ravel()], axis=1), dtype=torch.float32)
y_train_2d = torch.tensor(Z.ravel(), dtype=torch.float32).unsqueeze(1)

mlp_model = SimpleMLP()
reckan_model = RecKAN(degree=8, normalize=True)

criterion = nn.MSELoss()
optimizer_mlp = torch.optim.Adam(mlp_model.parameters(), lr=0.01)

# --- separate LR: recurrence coefficients (a..e) move much slower than
# the per-edge linear combination weights (coeffs), since a..e sit inside
# a recursive chain and are far more sensitive ---
base_lr = 0.01
recurrence_lr_scale = 0.1
optimizer_reckan = torch.optim.Adam([
    {'params': reckan_model.coeff_parameters(), 'lr': base_lr},
    {'params': reckan_model.recurrence_parameters(), 'lr': base_lr * recurrence_lr_scale},
])

epochs = 4000
warmup_epochs = 200   # keep a..e frozen at their Chebyshev-like init at first
mlp_losses, reckan_losses = [], []

for epoch in range(epochs):
    reckan_model.set_recurrence_trainable(epoch >= warmup_epochs)

    optimizer_mlp.zero_grad()
    outputs_mlp = mlp_model(x_train_2d)
    loss_mlp = criterion(outputs_mlp, y_train_2d)
    loss_mlp.backward()
    optimizer_mlp.step()

    optimizer_reckan.zero_grad()
    outputs_reckan = reckan_model(x_train_2d)
    loss_reckan = criterion(outputs_reckan, y_train_2d)
    loss_reckan.backward()
    torch.nn.utils.clip_grad_norm_(reckan_model.parameters(), max_norm=1.0)
    optimizer_reckan.step()

    if epoch % 100 == 0:
        mlp_losses.append(loss_mlp.item())
        reckan_losses.append(loss_reckan.item())
        tag = ' [recurrence frozen]' if epoch < warmup_epochs else ''
        print(f'Epoch {epoch+1}/{epochs}, MLP Loss: {loss_mlp.item():.4f}, '
              f'RecKAN Loss: {loss_reckan.item():.4f}{tag}')

print('\nLearned recurrence parameters (a,b,c,d,e):')
b = reckan_model.basis
print(f'a={b.a.item():.4f}  b={b.b.item():.4f}  c={b.c.item():.4f}  '
      f'd={b.d.item():.4f}  e={b.e.item():.4f}')

# ---------------------------------------------------------------------------
# Test + plot
# ---------------------------------------------------------------------------
x_test = np.linspace(0, 2, 400)
y_test = np.linspace(0, 2, 400)
X_test, Y_test = np.meshgrid(x_test, y_test)
Z_test = fractal_function(X_test, Y_test)
x_test_2d = torch.tensor(np.stack([X_test.ravel(), Y_test.ravel()], axis=1), dtype=torch.float32)

with torch.no_grad():
    y_pred_mlp = mlp_model(x_test_2d).numpy()
    y_pred_reckan = reckan_model(x_test_2d).numpy()

fig = plt.figure(figsize=(18, 6))
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(X_test, Y_test, y_pred_mlp.reshape(X_test.shape), cmap='viridis', alpha=0.85)
ax1.set_title('MLP Predictions')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

ax2 = fig.add_subplot(132, projection='3d')
ax2.plot_surface(X_test, Y_test, y_pred_reckan.reshape(X_test.shape), cmap='magma', alpha=0.85)
ax2.set_title('RecKAN (learnable recurrence) Predictions')
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')

ax3 = fig.add_subplot(133, projection='3d')
ax3.plot_surface(X_test, Y_test, Z_test, cmap='coolwarm', alpha=0.85)
ax3.set_title('Original Fractal Function (noisy)')
ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.set_zlabel('Z')

plt.tight_layout()
plt.savefig('/home/claude/fractal_2d_comparison.png', dpi=130)

plt.figure(figsize=(8, 5))
epochs_axis = np.arange(0, epochs, 100)
plt.plot(epochs_axis, mlp_losses, label='MLP loss')
plt.plot(epochs_axis, reckan_losses, label='RecKAN loss')
plt.yscale('log')
plt.xlabel('epoch'); plt.ylabel('MSE loss (log scale)')
plt.legend(); plt.title('2D fit training loss')
plt.tight_layout()
plt.savefig('/home/claude/fractal_2d_loss.png', dpi=130)

print('\nFinal MLP loss:', mlp_losses[-1])
print('Final RecKAN loss:', reckan_losses[-1])